# Importar ficheiros INF001 (Informe horas entrega) para SQLite

Notebook unico com todo o processo:

1. Preparar a base de dados (tabelas + indice unico anti-duplicados)
2. Ler os ficheiros `.xls` da pasta `entregas`
3. Inserir os dados, sem duplicar linhas

Basta correr todas as celulas por ordem (Kernel -> Run All).
Podes correr este notebook sempre que houver ficheiros novos na pasta -
os dados ja importados nao serao duplicados, mesmo que um ficheiro novo
repita linhas que ja vieram de outro ficheiro anterior.

## Configuracao

In [ ]:
import os
import glob
import platform
import sqlite3
import xml.etree.ElementTree as ET
from datetime import datetime

if platform.system() == "Windows":
    DB_PATH = r"C:\Users\LISARR\Documents\python\01.Financeiro\inform_27.db"
    PASTA_FICHEIROS = r"C:\Users\LISARR\Documents\python\02.Pontualidade\entregas"
elif platform.system() == "Darwin":
    DB_PATH = "/Volumes/RR/DB/inform_27.db"
    PASTA_FICHEIROS = "/Volumes/RR/DB/entregas"
else:
    DB_PATH = "inform_27.db"
    PASTA_FICHEIROS = "entregas"

print("DB_PATH:", DB_PATH)
print("PASTA_FICHEIROS:", PASTA_FICHEIROS)

In [ ]:
# Colunas do ficheiro INF001: nome_sql -> nome original da coluna no Excel
COLUNAS_ENTREGAS = {
    "PROPIETARIO_DT": "Propietario DT",
    "RECORRIDO_CAMION": "Recorrido Cami\u00f3n",
    "DESTINO": "Destino",
    "HORA_TEORICA_LLEGADA": "Hora te\u00f3rica Llegada",
    "HORA_REAL_LLEGADA": "Hora Real Llegada",
    "HORA_SALIDA_DESTINO": "Hora Salida de Destino",
    "RETRASO_LLEGADA": "Retraso Llegada",
    "TIEMPO_ESPERA": "Tiempo de Espera",
    "NUM_UT": "N\u00fam. UT",
    "FECHA_TEORICA_LLEGADA": "Fecha Te\u00f3rica Llegada",
    "FECHA_REAL_LLEGADA": "Fecha Real Llegada",
    "TEMPERATURA": "Temperatura",
    "EMPRESA_TRANSPORTE": "Empresa de Transporte",
    "MATRICULA_REMOLQUE": "Matr\u00edcula Remolque",
    "MOTIVO": "Motivo",
    "OBSERVACIONES": "Observaciones",
}

COLUNA_ORIGEM = "ficheiro_origem"
ANOS = ("2025", "2026")

# Colunas (nomes sql) que identificam uma linha como unica. Usa TODAS
# as colunas - uma linha so e considerada duplicada se for igual em
# tudo. Usadas pelo indice unico para nunca deixar a mesma entrega ser
# inserida 2 vezes.
CHAVE_UNICA = tuple(COLUNAS_ENTREGAS.keys())

## Passo 1 — Preparar a base de dados

In [ ]:
def preparar_base_dados(con):
    """Cria a base de dados (se nao existir), as tabelas entregas_<ano>
    e o indice unico que impede duplicados. Usa COALESCE(..., '') porque
    o SQLite trata cada NULL como diferente de outro NULL - sem isto,
    linhas com campos da chave em branco nunca seriam consideradas
    duplicadas. O indice e sempre recriado (DROP + CREATE) para garantir
    que reflete a CHAVE_UNICA atual, mesmo que ja existisse com outra
    definicao de uma versao anterior deste notebook."""
    colunas_sql = ",\n        ".join(f'"{c}" TEXT' for c in COLUNAS_ENTREGAS)
    chave_sql = ", ".join(f'COALESCE("{c}", \'\')' for c in CHAVE_UNICA)

    cur = con.cursor()
    for ano in ANOS:
        cur.execute(f'''
            CREATE TABLE IF NOT EXISTS entregas_{ano} (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                {colunas_sql},
                "{COLUNA_ORIGEM}" TEXT
            )
        ''')
        cur.execute(f'DROP INDEX IF EXISTS idx_entregas_{ano}_chave')
        cur.execute(
            f'CREATE UNIQUE INDEX idx_entregas_{ano}_chave '
            f'ON entregas_{ano}({chave_sql})'
        )
    con.commit()


pasta_db = os.path.dirname(DB_PATH)
if pasta_db and not os.path.exists(pasta_db):
    os.makedirs(pasta_db, exist_ok=True)

con = sqlite3.connect(DB_PATH)
preparar_base_dados(con)
con.close()
print("Base de dados, tabelas e indice unico prontos.")

## Passo 2 — Ler os ficheiros da pasta

In [ ]:
def listar_ficheiros_excel(pasta):
    """Procura ficheiros .xls (tambem em subpastas), sem duplicar por
    causa de maiusculas/minusculas no caminho."""
    encontrados = glob.glob(os.path.join(pasta, "**", "*.xls"), recursive=True)
    vistos = set()
    ficheiros = []
    for caminho in encontrados:
        chave = os.path.normcase(os.path.abspath(caminho))
        if chave not in vistos:
            vistos.add(chave)
            ficheiros.append(caminho)
    return sorted(ficheiros)


ficheiros = listar_ficheiros_excel(PASTA_FICHEIROS)
print(f"Ficheiros .xls encontrados: {len(ficheiros)}")

In [ ]:
# ==========================
# 1. Leitura Excel XML
# ==========================
NS = {"ss": "urn:schemas-microsoft-com:office:spreadsheet"}
SS_INDEX = "{urn:schemas-microsoft-com:office:spreadsheet}Index"


def ler_linhas_xml_spreadsheet(caminho_ficheiro, min_colunas_cabecalho=5):
    """Le todas as worksheets validas de um Excel XML Spreadsheet 2003."""
    tree = ET.parse(caminho_ficheiro)
    root = tree.getroot()

    worksheets = root.findall("ss:Worksheet", NS)
    if not worksheets:
        raise ValueError("Nao foi encontrada nenhuma 'Worksheet' no ficheiro.")

    def extrair_linha(linha_xml):
        valores = []
        proximo_indice = 1

        for cell in linha_xml.findall("ss:Cell", NS):
            idx = cell.get(SS_INDEX)
            idx = int(idx) if idx is not None else proximo_indice

            while len(valores) < idx - 1:
                valores.append(None)

            data_el = cell.find("ss:Data", NS)
            valores.append(data_el.text if data_el is not None else None)
            proximo_indice = idx + 1

        return valores

    cabecalho_referencia = None
    linhas_totais = []

    for worksheet in worksheets:
        tabela = worksheet.find("ss:Table", NS)
        if tabela is None:
            continue

        linhas_xml = tabela.findall("ss:Row", NS)
        if not linhas_xml:
            continue

        idx_cabecalho = None
        cabecalho = None

        for i, linha_xml in enumerate(linhas_xml):
            valores = extrair_linha(linha_xml)
            preenchidas = sum(v not in (None, "") for v in valores)

            if preenchidas >= min_colunas_cabecalho:
                idx_cabecalho = i
                cabecalho = [
                    str(v).strip() if v else f"COLUNA_{j + 1}"
                    for j, v in enumerate(valores)
                ]
                break

        if idx_cabecalho is None:
            continue

        # So agrega folhas com o mesmo cabecalho de dados
        if cabecalho_referencia is None:
            cabecalho_referencia = cabecalho
        elif cabecalho != cabecalho_referencia:
            continue

        n_colunas = len(cabecalho_referencia)

        for linha_xml in linhas_xml[idx_cabecalho + 1:]:
            valores = extrair_linha(linha_xml)

            if len(valores) < n_colunas:
                valores += [None] * (n_colunas - len(valores))
            elif len(valores) > n_colunas:
                valores = valores[:n_colunas]

            linhas_totais.append(valores)

    if cabecalho_referencia is None:
        raise ValueError("Nao foi encontrada uma linha de cabecalho valida.")

    return cabecalho_referencia, linhas_totais


print("Funcao ler_linhas_xml_spreadsheet() pronta.")


## Passo 3 — Inserir os dados (sem duplicar)

In [ ]:
def validar_data(valor):
    """Valida uma data DD/MM/AAAA e devolve o objeto datetime."""
    if not valor:
        return None

    texto = str(valor).strip()
    if not texto or texto.upper() == "N/D":
        return None

    try:
        return datetime.strptime(texto, "%d/%m/%Y")
    except ValueError:
        return None


def extrair_ano(mapa):
    """Devolve o ano de Fecha Teorica Llegada quando a data e valida."""
    valor = mapa.get(COLUNAS_ENTREGAS["FECHA_TEORICA_LLEGADA"])
    data = validar_data(valor)
    return str(data.year) if data else None


def montar_linha(mapa, nome_ficheiro):
    """Constroi a linha a inserir. Remove espacos a mais em todos os
    campos de texto - sem isto, a mesma entrega podia ficar com uma
    chave diferente entre ficheiros (ex: 'PT-1' vs 'PT-1 ') e o indice
    unico deixava de a reconhecer como duplicada."""
    valores = []
    for nome_original in COLUNAS_ENTREGAS.values():
        valor = mapa.get(nome_original)
        if isinstance(valor, str):
            valor = valor.strip()
        valores.append(valor)
    return valores + [nome_ficheiro]


cols_sql = ", ".join(f'"{c}"' for c in COLUNAS_ENTREGAS)
placeholders = ", ".join(["?"] * len(COLUNAS_ENTREGAS))
sql_insercao = {
    ano: (
        f'INSERT OR IGNORE INTO entregas_{ano} ({cols_sql}, "{COLUNA_ORIGEM}") '
        f'VALUES ({placeholders}, ?)'
    )
    for ano in ANOS
}

print("Funcoes e instrucoes SQL prontas.")

In [ ]:
# ==========================
# 1. Preparar importacao
# ==========================
contagem_novas = {ano: 0 for ano in ANOS}
total_duplicadas = 0
total_sem_data = 0
total_processados = 0
total_erros = 0

con = sqlite3.connect(DB_PATH)
cur = con.cursor()

# ==========================
# 2. Processar ficheiros
# ==========================
for i, caminho in enumerate(ficheiros, start=1):
    nome_ficheiro = os.path.basename(caminho)

    try:
        cabecalho, linhas = ler_linhas_xml_spreadsheet(caminho)

        novas_ficheiro = {ano: 0 for ano in ANOS}
        duplicadas_ficheiro = 0
        sem_data_ficheiro = 0

        for valores in linhas:
            if all(v is None or str(v).strip() == "" for v in valores):
                continue

            mapa = dict(zip(cabecalho, valores))
            ano = extrair_ano(mapa)

            if ano not in ANOS:
                sem_data_ficheiro += 1
                continue

            linha = montar_linha(mapa, nome_ficheiro)
            cur.execute(sql_insercao[ano], linha)

            if cur.rowcount == 1:
                novas_ficheiro[ano] += 1
            else:
                duplicadas_ficheiro += 1

        # Confirma apenas este ficheiro
        con.commit()
        total_processados += 1

        for ano in ANOS:
            contagem_novas[ano] += novas_ficheiro[ano]

        total_duplicadas += duplicadas_ficheiro
        total_sem_data += sem_data_ficheiro

        agora = datetime.now().isoformat(timespec="seconds")
        print(
            f"[{i}/{len(ficheiros)}] {nome_ficheiro} -> "
            f"{novas_ficheiro['2025']} novas 2025, "
            f"{novas_ficheiro['2026']} novas 2026, "
            f"{duplicadas_ficheiro} ja existiam, "
            f"{sem_data_ficheiro} sem data valida | {agora}"
        )

    except Exception as erro:
        # Reverte apenas o ficheiro atual
        con.rollback()
        total_erros += 1
        print(f"[{i}/{len(ficheiros)}] [ERRO] '{nome_ficheiro}': {erro}")

# ==========================
# 3. Fechar e resumir
# ==========================
con.close()

print("=" * 70)
print("RESUMO FINAL - ENTREGAS")
print("=" * 70)
print(f"Ficheiros encontrados:                 {len(ficheiros)}")
print(f"Ficheiros processados:                 {total_processados}")
print(f"Ficheiros com erro:                    {total_erros}")
print(f"Linhas novas inseridas em 2025:        {contagem_novas['2025']}")
print(f"Linhas novas inseridas em 2026:        {contagem_novas['2026']}")
print(f"Linhas ja existentes (nao inseridas):  {total_duplicadas}")
print(f"Linhas sem data valida:                {total_sem_data}")
print("=" * 70)


In [ ]:
# ==========================
# Validacao final: comparar todos os ficheiros da pasta com o que esta
# na BD, agrupado por mes. Usa a mesma CHAVE_UNICA da importacao.
# ==========================
import pandas as pd

def obter_chaves_bd(con):
    """Devolve, por ano, o conjunto de chaves ja existentes na BD."""
    colunas_sql = ", ".join(f'COALESCE("{c}", \'\')' for c in CHAVE_UNICA)
    cur = con.cursor()
    chaves = {}
    for ano in ANOS:
        cur.execute(f'SELECT {colunas_sql} FROM entregas_{ano}')
        chaves[ano] = set(cur.fetchall())
    return chaves


def validar_ficheiros(ficheiros, chaves_bd):
    colunas_ordem = list(COLUNAS_ENTREGAS.keys())
    chaves_ficheiros_por_mes = {}  # mes_ano -> {"ano": ano, "chaves": set()}

    for caminho in ficheiros:
        cabecalho, linhas = ler_linhas_xml_spreadsheet(caminho)
        for valores in linhas:
            if all(v is None or str(v).strip() == "" for v in valores):
                continue
            mapa = dict(zip(cabecalho, valores))
            ano = extrair_ano(mapa)
            if ano not in ANOS:
                continue

            fecha = mapa.get(COLUNAS_ENTREGAS["FECHA_TEORICA_LLEGADA"])
            data_valida = validar_data(fecha)
            if data_valida is None:
                continue

            mes_ano = data_valida.strftime("%m/%Y")

            linha = montar_linha(mapa, os.path.basename(caminho))
            chave = tuple(str(linha[colunas_ordem.index(c)] or "") for c in CHAVE_UNICA)

            entrada = chaves_ficheiros_por_mes.setdefault(mes_ano, {"ano": ano, "chaves": set()})
            entrada["chaves"].add(chave)

    resultado = []
    for mes_ano, dados in chaves_ficheiros_por_mes.items():
        chaves_bd_ano = chaves_bd[dados["ano"]]
        total = len(dados["chaves"])
        na_bd = len(dados["chaves"] & chaves_bd_ano)
        resultado.append({
            "Mes/Ano": mes_ano,
            "Nos ficheiros": total,
            "Na BD": na_bd,
            "Em falta": total - na_bd,
        })

    df = pd.DataFrame(resultado)
    df["_ord"] = df["Mes/Ano"].apply(lambda m: m.split("/")[::-1])
    return df.sort_values("_ord").drop(columns="_ord").reset_index(drop=True)


con = sqlite3.connect(DB_PATH)
chaves_bd = obter_chaves_bd(con)
con.close()

validar_ficheiros(ficheiros, chaves_bd)

In [ ]:
# ==========================
# Diagnostico: onde esta a falhar a insercao de agosto/2026?
# ==========================
import pandas as pd

# 1. Confirmar que os ficheiros estao na pasta que o notebook realmente le
ficheiros_atual = listar_ficheiros_excel(PASTA_FICHEIROS)
print(f"Ficheiros na pasta agora: {len(ficheiros_atual)}")
for f in ficheiros_atual:
    print(" -", os.path.basename(f))

# 2. Contar linhas de agosto/2026 em cada ficheiro da pasta
print("\nLinhas de 08/2026 por ficheiro:")
for caminho in ficheiros_atual:
    cabecalho, linhas = ler_linhas_xml_spreadsheet(caminho)
    idx_fecha = cabecalho.index(COLUNAS_ENTREGAS["FECHA_TEORICA_LLEGADA"])
    n_agosto = sum(
        1 for l in linhas
        if l[idx_fecha] and "/08/2026" in str(l[idx_fecha])
    )
    if n_agosto:
        print(f"  {os.path.basename(caminho)}: {n_agosto}")

# 3. Ver o que ja esta na BD para agosto/2026
con = sqlite3.connect(DB_PATH)
df_bd = pd.read_sql_query(
    "SELECT FECHA_TEORICA_LLEGADA, ficheiro_origem, COUNT(*) as n "
    "FROM entregas_2026 WHERE FECHA_TEORICA_LLEGADA LIKE '%/08/2026' "
    "GROUP BY FECHA_TEORICA_LLEGADA, ficheiro_origem "
    "ORDER BY FECHA_TEORICA_LLEGADA",
    con,
)
con.close()

print("\nLinhas de 08/2026 ja na BD:")
df_bd